# LeetCode 3405 — Count the Number of Arrays with K Matching Adjacent Elements bằng Python

- Link: https://leetcode.com/problems/count-the-number-of-arrays-with-k-matching-adjacent-elements/
- Độ khó: **Hard**
- Chủ đề: **Tổ hợp** (chọn vị trí "bằng nhau"), **lũy thừa nhanh modulo**, **giai thừa + nghịch đảo Fermat**, đối chiếu bằng **DP**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countGoodArrays(self, n: int, m: int, k: int) -> int:
```

## 1. Mục tiêu học

1. Nhìn mảng như dãy `n − 1` **"khe"** giữa hai phần tử kề nhau; mỗi khe là "bằng" hoặc "khác".
2. Đếm: chọn `k` khe "bằng" (`C(n−1, k)`), phần tử đầu `m` cách, mỗi khe "khác" `m − 1` cách ⇒ `m · (m−1)^{n−1−k} · C(n−1, k)`.
3. Tính tổ hợp modulo số nguyên tố bằng **giai thừa và nghịch đảo** (định lý Fermat nhỏ).
4. Kiểm chứng công thức bằng **DP** `dp[i][j]` (độ dài `i`, có `j` cặp bằng) và vét cạn.
5. Cẩn thận các biên: `m = 1`, `k = n − 1`, `n = 1`.

## 2. Đề bài & ví dụ

Đếm số mảng `arr` độ dài `n`, phần tử trong `[1, m]`, có **đúng `k`** chỉ số `i` (`1 ≤ i < n`) với `arr[i−1] == arr[i]`. Kết quả mod `10^9 + 7`.

**Ví dụ 1:** `n = 3, m = 2, k = 1` → `4`. **Ví dụ 2:** `n = 4, m = 2, k = 2` → `6`. **Ví dụ 3:** `n = 5, m = 2, k = 0` → `2`.

**Ràng buộc:** `1 ≤ n, m ≤ 10^5`, `0 ≤ k ≤ n − 1`.

In [1]:
MOD = 10 ** 9 + 7
examples = [(3, 2, 1, 4), (4, 2, 2, 6), (5, 2, 0, 2)]
for n, m, k, exp in examples:
    print(n, m, k, '-> kỳ vọng', exp)

3 2 1 -> kỳ vọng 4
4 2 2 -> kỳ vọng 6
5 2 0 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Mã hóa mảng bằng (phần tử đầu, kiểu từng khe)

Cho phần tử đầu và với mỗi khe `i` biết "bằng" (giá trị giữ nguyên) hay "khác" (chọn một trong `m − 1` giá trị còn lại) ⇒ xác định duy nhất mảng. Ngược lại mỗi mảng cho đúng một mã như vậy.

In [2]:
import itertools

def encode(arr):
    return arr[0], ''.join('=' if arr[i] == arr[i - 1] else '≠' for i in range(1, len(arr)))

for arr in [(1, 1, 2), (1, 2, 2), (2, 1, 1), (2, 2, 1)]:
    print(arr, '->', encode(arr))

(1, 1, 2) -> (1, '=≠')
(1, 2, 2) -> (1, '≠=')
(2, 1, 1) -> (2, '≠=')
(2, 2, 1) -> (2, '=≠')


### 3.2. Lũy thừa nhanh và tổ hợp modulo

`pow(a, e, MOD)` của Python dùng bình phương liên tiếp, `O(log e)`. `C(N, r) = N! · (r!)^{-1} · ((N−r)!)^{-1}` với nghịch đảo `x^{MOD−2}`.

In [3]:
from math import comb

def comb_mod(N, r):
    if r < 0 or r > N:
        return 0
    num = den = 1
    for i in range(r):
        num = num * (N - i) % MOD
        den = den * (i + 1) % MOD
    return num * pow(den, MOD - 2, MOD) % MOD

print(pow(3, 10 ** 18, MOD), comb_mod(10, 4), comb(10, 4), comb_mod(10 ** 5, 31337) == comb(10 ** 5, 31337) % MOD)

246336683 210 210 True


### 3.3. Kiểm tra tổng bằng định lý nhị thức

Cộng đáp án trên mọi `k` phải ra **tổng số mảng** `m^n`. Thật vậy, `Σ_k C(n−1, k) · 1^k · (m−1)^{n−1−k} = (1 + (m − 1))^{n−1} = m^{n−1}`, nhân thêm `m` cho phần tử đầu ⇒ `m^n`. Đây là một phép kiểm tra nhanh rất hữu ích khi tự suy ra công thức đếm: nếu tổng không khớp, chắc chắn công thức sai.

In [4]:
for n, m in [(1, 5), (3, 2), (6, 4), (10, 7)]:
    total = sum(comb(n - 1, k) * m * (m - 1) ** (n - 1 - k) for k in range(n))
    print(f"n={n}, m={m}: Σ_k = {total}, m^n = {m ** n}, khớp: {total == m ** n}")

n=1, m=5: Σ_k = 5, m^n = 5, khớp: True
n=3, m=2: Σ_k = 8, m^n = 8, khớp: True
n=6, m=4: Σ_k = 4096, m^n = 4096, khớp: True
n=10, m=7: Σ_k = 282475249, m^n = 282475249, khớp: True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 (`n = 3, m = 2, k = 1`): có `n − 1 = 2` khe, chọn 1 khe "bằng": `C(2, 1) = 2` cách (`=≠` hoặc `≠=`). Phần tử đầu: 2 cách. Khe "khác": `m − 1 = 1` cách. Tổng `2 · 2 · 1 = 4` ✓.

Ví dụ 3 (`k = 0`): mọi khe "khác" ⇒ `m · (m−1)^{n−1} = 2 · 1 = 2` ✓.

In [5]:
for n, m, k, exp in examples:
    print(f"C({n - 1},{k}) · {m} · {m - 1}^{n - 1 - k} = {comb(n - 1, k) * m * (m - 1) ** (n - 1 - k)} | kỳ vọng {exp}")

C(2,1) · 2 · 1^1 = 4 | kỳ vọng 4
C(3,2) · 2 · 1^1 = 6 | kỳ vọng 6
C(4,0) · 2 · 1^4 = 2 | kỳ vọng 2


## 5. Cách 1 — Brute Force: liệt kê mọi mảng

In [6]:
def count_brute(n, m, k):
    c = 0
    for arr in itertools.product(range(1, m + 1), repeat=n):
        if sum(arr[i] == arr[i - 1] for i in range(1, n)) == k:
            c += 1
    return c % MOD

for n, m, k, exp in examples:
    print(count_brute(n, m, k), '| kỳ vọng', exp)

4 | kỳ vọng 4
6 | kỳ vọng 6
2 | kỳ vọng 2


**Độ phức tạp:** `O(m^n · n)` — chỉ cho `m^n ≤ 10^5`.

## 6. Ý tưởng thuật toán / bước đột phá

Song ánh ở mục 3.1 cho:

`đáp án = m · (m − 1)^{n−1−k} · C(n − 1, k)  (mod 10^9 + 7)`.

Tính `C(n−1, k)` bằng giai thừa modulo (`O(n)`), lũy thừa bằng `pow` (`O(log n)`).

Vì sao được **nhân** các lựa chọn: sau khi cố định mẫu khe, lựa chọn ở mỗi khe "khác" chỉ phụ thuộc giá trị **ngay trước** nó (phải khác giá trị đó), và luôn có đúng `m − 1` giá trị hợp lệ bất kể giá trị trước là gì. Số lựa chọn không phụ thuộc lịch sử ⇒ quy tắc nhân áp dụng được.

Một chi tiết nhỏ: khi `m = 1` mọi khe phải là `=` (không có giá trị khác), nên đáp án là `1` nếu `k = n − 1` và `0` nếu ngược lại. Công thức xử lý tự động vì `0^0 = 1` và `0^e = 0` với `e > 0` — `pow(0, 0, MOD)` trong Python trả về `1`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP `dp[i][j]`: độ dài `i`, đúng `j` cặp bằng

`dp[1][0] = m`; thêm phần tử: bằng phần tử trước (`dp[i][j] → dp[i+1][j+1]`, 1 cách) hoặc khác (`→ dp[i+1][j]`, `m − 1` cách). `O(n · k)` — dùng để kiểm chứng.

In [7]:
def count_dp(n, m, k):
    dp = [0] * (k + 1)
    dp[0] = m % MOD
    for _ in range(n - 1):
        new = [0] * (k + 1)
        for j, v in enumerate(dp):
            if v:
                new[j] = (new[j] + v * (m - 1)) % MOD
                if j + 1 <= k:
                    new[j + 1] = (new[j + 1] + v) % MOD
        dp = new
    return dp[k]

for n, m, k, exp in examples:
    print(count_dp(n, m, k), '| kỳ vọng', exp)

4 | kỳ vọng 4
6 | kỳ vọng 6
2 | kỳ vọng 2


### 7.2. Cách 3 — Công thức + giai thừa modulo (lời giải chính)

In [8]:
def count_formula(n, m, k):
    N = n - 1
    fact = [1] * (N + 1)
    for i in range(1, N + 1):
        fact[i] = fact[i - 1] * i % MOD
    c = fact[N] * pow(fact[k] * fact[N - k] % MOD, MOD - 2, MOD) % MOD
    return m * pow(m - 1, N - k, MOD) % MOD * c % MOD

for n, m, k, exp in examples:
    print(count_formula(n, m, k), '| kỳ vọng', exp)

4 | kỳ vọng 4
6 | kỳ vọng 6
2 | kỳ vọng 2


### 7.3. Cách 4 — Công thức với `math.comb` (số nguyên lớn) rồi lấy mod

Python tính `comb(n − 1, k)` chính xác; với `n ≤ 10^5` vẫn nhanh. Ngắn gọn nhưng phụ thuộc số lớn.

In [9]:
def count_bigint(n, m, k):
    return comb(n - 1, k) % MOD * m % MOD * pow(m - 1, n - 1 - k, MOD) % MOD

for n, m, k, exp in examples:
    print(count_bigint(n, m, k), '| kỳ vọng', exp)

4 | kỳ vọng 4
6 | kỳ vọng 6
2 | kỳ vọng 2


## 8. TRACE từng bước

Trace DP (cách 2) với `n = 4, m = 3, k = 1` và so với công thức.

In [10]:
def trace(n, m, k):
    dp = [0] * (n)
    dp[0] = m
    print(f"độ dài 1: dp = {dp[:k + 1]}")
    for length in range(2, n + 1):
        new = [0] * n
        for j in range(n):
            if dp[j]:
                new[j] += dp[j] * (m - 1)
                if j + 1 < n:
                    new[j + 1] += dp[j]
        dp = new
        print(f"độ dài {length}: dp = {dp[:n]}  (dp[j] = số mảng có j cặp bằng)")
    print('dp[k] =', dp[k], '| công thức:', comb(n - 1, k) * m * (m - 1) ** (n - 1 - k))

trace(4, 3, 1)

độ dài 1: dp = [3, 0]
độ dài 2: dp = [6, 3, 0, 0]  (dp[j] = số mảng có j cặp bằng)
độ dài 3: dp = [12, 12, 3, 0]  (dp[j] = số mảng có j cặp bằng)
độ dài 4: dp = [24, 36, 18, 3]  (dp[j] = số mảng có j cặp bằng)
dp[k] = 36 | công thức: 36


Hàng cuối của DP chính là `m · C(n−1, j) · (m−1)^{n−1−j}` cho mọi `j` — khai triển nhị thức của `m · (1 + (m−1))^{n−1}`; tổng hàng = `m^n` (mọi mảng).

### 8.2. Trace theo mẫu khe

Liệt kê mọi chuỗi khe có đúng `k` dấu `=` với `n = 4, m = 3, k = 2`, và số mảng sinh ra từ mỗi mẫu: `m · (m−1)^{số dấu ≠}`.

In [11]:
n, m, k = 4, 3, 2
total = 0
for pos in itertools.combinations(range(n - 1), k):
    pattern = ''.join('=' if i in pos else '≠' for i in range(n - 1))
    ways = m * (m - 1) ** (n - 1 - k)
    total += ways
    sample = [1]
    for ch in pattern:
        sample.append(sample[-1] if ch == '=' else (sample[-1] % m) + 1)
    print(f"mẫu {pattern}: {ways} mảng, ví dụ {sample}")
print('tổng =', total, '| vét cạn =', count_brute(n, m, k))

mẫu ==≠: 6 mảng, ví dụ [1, 1, 1, 2]
mẫu =≠=: 6 mảng, ví dụ [1, 1, 2, 2]
mẫu ≠==: 6 mảng, ví dụ [1, 2, 2, 2]
tổng = 18 | vét cạn = 18


Mỗi mẫu đóng góp **cùng một** số mảng, nên đáp án = (số mẫu) × (số mảng mỗi mẫu) — chính là tích `C(n−1, k) · m · (m−1)^{n−1−k}`.

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def countGoodArrays(self, n: int, m: int, k: int) -> int:
        MOD = 10 ** 9 + 7
        N = n - 1
        fact = [1] * (N + 1)
        for i in range(1, N + 1):
            fact[i] = fact[i - 1] * i % MOD
        c = fact[N] * pow(fact[k], MOD - 2, MOD) % MOD * pow(fact[N - k], MOD - 2, MOD) % MOD
        return m * pow(m - 1, N - k, MOD) % MOD * c % MOD

for n, m, k, exp in examples:
    got = Solution().countGoodArrays(n, m, k)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
6 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Song ánh (3.1):** mảng ↔ (giá trị đầu, chuỗi `n − 1` ký hiệu `=`/`≠`, và với mỗi `≠` một giá trị mới khác giá trị trước).
2. **Đếm:** số chuỗi ký hiệu có đúng `k` dấu `=` là `C(n−1, k)`; giá trị đầu `m` cách; mỗi `≠` có `m − 1` cách, độc lập ⇒ nhân.
3. **Modulo:** `MOD` nguyên tố, `n − 1 < MOD` ⇒ giai thừa khả nghịch; `pow(0, 0, MOD) = 1` xử lý đúng trường hợp `m = 1, k = n − 1`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê | `O(m^n · n)` | `O(n)` |
| 2. DP | `O(n · k)` | `O(k)` |
| 3. Công thức + giai thừa | `O(n + log MOD)` | `O(n)` |
| 4. `math.comb` | phụ thuộc số lớn, nhanh với `n ≤ 10^5` | số lớn |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [13]:
impls = {'dp': count_dp, 'formula': count_formula, 'bigint': count_bigint,
         'submit': lambda n, m, k: Solution().countGoodArrays(n, m, k)}
edge = [(1, 1, 0, 1), (1, 7, 0, 7), (3, 1, 2, 1), (3, 1, 0, 0), (2, 5, 1, 5), (2, 5, 0, 20)]
for n, m, k, exp in examples + edge:
    assert count_brute(n, m, k) == exp, (n, m, k, count_brute(n, m, k))
    for name, f in impls.items():
        assert f(n, m, k) == exp, (name, n, m, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu với vét cạn (mọi bộ nhỏ) và với DP (bộ vừa)

In [14]:
import random
cnt = 0
for n in range(1, 7):
    for m in range(1, 5):
        if m ** n > 5000:
            continue
        for k in range(n):
            exp = count_brute(n, m, k)
            for name, f in impls.items():
                assert f(n, m, k) == exp, (name, n, m, k)
            cnt += 1
rng = random.Random(3405)
for _ in range(200):
    n = rng.randint(1, 200); m = rng.randint(1, 10 ** 5); k = rng.randint(0, n - 1)
    assert count_dp(n, m, k) == Solution().countGoodArrays(n, m, k)
print('Khớp vét cạn trên', cnt, 'bộ nhỏ; khớp DP trên 200 bộ vừa.')

Khớp vét cạn trên 84 bộ nhỏ; khớp DP trên 200 bộ vừa.


### 12.3. Ca lớn

In [15]:
import time
for n, m, k in [(10 ** 5, 10 ** 5, 0), (10 ** 5, 10 ** 5, 50000), (10 ** 5, 2, 99999), (10 ** 5, 1, 99998)]:
    t = time.time(); a = Solution().countGoodArrays(n, m, k); t1 = time.time() - t
    t = time.time(); b = count_bigint(n, m, k); t2 = time.time() - t
    assert a == b
    print(f"n={n} m={m} k={k}: {a} | giai thừa {t1:.3f}s | math.comb {t2:.3f}s")

n=100000 m=100000 k=0: 406242317 | giai thừa 0.007s | math.comb 0.000s


n=100000 m=100000 k=50000: 509990004 | giai thừa 0.007s | math.comb 0.090s
n=100000 m=2 k=99999: 2 | giai thừa 0.006s | math.comb 0.000s
n=100000 m=1 k=99998: 0 | giai thừa 0.007s | math.comb 0.000s


### 12.4. Kiểm tra các tính chất của đáp án

Ba tính chất có thể kiểm bằng máy mà không cần vét cạn:

1. **Tổng theo `k` bằng `m^n`** (mục 3.3) — kiểm trên modulo với `n` lớn.
2. **Tỉ số liên tiếp:** `f(k+1) / f(k) = (n−1−k) / ((k+1)(m−1))` (với `m ≥ 2`) — suy từ `C(n−1, k+1) / C(n−1, k)` và bớt một thừa số `m − 1`.
3. **`m = 2` đối xứng quanh** `k ↦ n − 1 − k` vì `(m − 1)^{…} = 1` và `C(n−1, k) = C(n−1, n−1−k)`.

In [16]:
n, m = 5000, 37
f = [Solution().countGoodArrays(n, m, k) for k in range(n)]
assert sum(f) % MOD == pow(m, n, MOD)
for k in range(0, n - 1, 499):
    lhs = f[k + 1] * (k + 1) % MOD * (m - 1) % MOD
    rhs = f[k] * (n - 1 - k) % MOD
    assert lhs == rhs
g = [Solution().countGoodArrays(300, 2, k) for k in range(300)]
assert g == g[::-1]
print('Tổng = m^n (mod p), tỉ số liên tiếp đúng, và m = 2 đối xứng: tất cả đúng.')

Tổng = m^n (mod p), tỉ số liên tiếp đúng, và m = 2 đối xứng: tất cả đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên hệ số `m` của phần tử đầu

In [17]:
def wrong_no_m(n, m, k):
    return comb(n - 1, k) * pow(m - 1, n - 1 - k, MOD) % MOD

print('sai :', wrong_no_m(3, 2, 1), '| đúng:', Solution().countGoodArrays(3, 2, 1))

sai : 2 | đúng: 4


**Vì sao sai:** phần tử đầu tiên không có "khe" trước nó nhưng vẫn có `m` lựa chọn.

### Lỗi 2 — Dùng `C(n, k)` (đếm trên `n` vị trí thay vì `n − 1` khe)

In [18]:
def wrong_comb_n(n, m, k):
    return comb(n, k) * m * pow(m - 1, n - 1 - k, MOD) % MOD

print('sai :', wrong_comb_n(4, 2, 2), '| đúng:', Solution().countGoodArrays(4, 2, 2))

sai : 12 | đúng: 6


**Vì sao sai:** cặp kề nhau có `n − 1` vị trí (`i = 1..n−1`).

### Lỗi 3 — Số mũ `n − k` thay vì `n − 1 − k`

In [19]:
def wrong_exponent(n, m, k):
    return comb(n - 1, k) * m * pow(m - 1, n - k, MOD) % MOD

print('sai :', wrong_exponent(5, 3, 0), '| đúng:', Solution().countGoodArrays(5, 3, 0))

sai : 96 | đúng: 48


**Vì sao sai:** số khe "khác" là `(n − 1) − k`.

### Lỗi 4 — Chia modulo bằng `//`

In [20]:
def wrong_division(n, m, k):
    N = n - 1
    fact = [1] * (N + 1)
    for i in range(1, N + 1):
        fact[i] = fact[i - 1] * i % MOD
    c = fact[N] // (fact[k] * fact[N - k] % MOD)               # SAI
    return m * pow(m - 1, N - k, MOD) % MOD * c % MOD

print('sai :', wrong_division(10 ** 5, 5, 777), '| đúng:', Solution().countGoodArrays(10 ** 5, 5, 777))

sai : 11842223 | đúng: 924524635


**Vì sao sai:** sau khi lấy mod, `fact[N]` không còn chia hết cho tích hai giai thừa kia. Phải nhân **nghịch đảo modulo**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đúng `k` cặp kề nhau bằng nhau" ⇒ chọn `k` khe trong `n − 1` khe.
- Đếm mảng theo quan hệ giữa phần tử kề ⇒ mã hóa bằng (phần tử đầu, kiểu khe).
- Tổ hợp modulo `10^9 + 7` ⇒ giai thừa + `pow(x, MOD − 2, MOD)`.
- Kiểm tra công thức bằng DP nhỏ trước khi nộp.
- Kiểm tra nhanh mọi công thức đếm bằng **tổng trên mọi trường hợp** (ở đây phải ra `m^n`) và vài giá trị nhỏ tính tay.

## 15. Biến thể của bài

1. **Ít nhất** `k` cặp bằng ⇒ cộng công thức cho `j = k..n−1`.
2. Mảng **vòng** (cả cặp `arr[n−1], arr[0]`) ⇒ cần bao hàm – loại trừ / DP theo giá trị đầu.
3. Đúng `k` cặp **khác** ⇒ đổi vai `k ↔ n − 1 − k`.
4. Nhiều truy vấn `(n, m, k)` ⇒ tiền xử lý giai thừa một lần tới `max n`.

**Liên hệ xác suất:** nếu chọn mảng ngẫu nhiên đều trong `m^n` mảng thì mỗi khe là "bằng" độc lập với xác suất `1/m`. Số khe "bằng" tuân theo **phân phối nhị thức** `Bin(n − 1, 1/m)`, nên `P(đúng k) = C(n−1, k) (1/m)^k (1 − 1/m)^{n−1−k}`. Nhân với `m^n` ra đúng công thức của bài — một cách nhớ công thức rất nhanh, và cũng cho kỳ vọng số cặp bằng là `(n − 1)/m`.

In [21]:
# Biến thể 1 và 4: ít nhất k cặp bằng; nhiều truy vấn với giai thừa dùng chung
MAXN = 2000
FACT = [1] * (MAXN + 1)
for i in range(1, MAXN + 1):
    FACT[i] = FACT[i - 1] * i % MOD
INVF = [1] * (MAXN + 1)
INVF[MAXN] = pow(FACT[MAXN], MOD - 2, MOD)
for i in range(MAXN, 0, -1):
    INVF[i - 1] = INVF[i] * i % MOD

def query(n, m, k):
    N = n - 1
    return m * pow(m - 1, N - k, MOD) % MOD * FACT[N] % MOD * INVF[k] % MOD * INVF[N - k] % MOD

def at_least(n, m, k):
    return sum(query(n, m, j) for j in range(k, n)) % MOD

assert all(query(n, m, k) == Solution().countGoodArrays(n, m, k) for n, m, k, _ in examples)
assert at_least(4, 2, 2) == sum(count_brute(4, 2, j) for j in range(2, 4)) % MOD
print('Biến thể: ít nhất 2 cặp bằng với n=4, m=2 ->', at_least(4, 2, 2))

Biến thể: ít nhất 2 cặp bằng với n=4, m=2 -> 8


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Liệt kê | mũ | Oracle |
| 2 | DP theo số cặp bằng | `O(nk)` | Kiểm chứng |
| 3 | Công thức + giai thừa | `O(n)` | **Nộp bài** |
| 4 | `math.comb` | nhanh trong Python | Phụ thuộc số lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 2, m = 3, k = 1`.

**Bài 2 (tính tay):** `n = 4, m = 3, k = 2`.

**Bài 3 (điền TODO):** viết `count_exact_different(n, m, d)` — đúng `d` cặp **khác** nhau.

**Bài 4 (tự viết lại):** tính `INVF` bằng công thức truy hồi `inv[i] = −(MOD // i) · inv[MOD % i]` thay cho Fermat.

**Bài 5 (mini challenge):** biến thể 2 — mảng **vòng** có đúng `k` cặp kề bằng nhau (kể cả cặp cuối–đầu), `n ≥ 2`.

In [22]:
def count_exact_different_todo(n, m, d):
    # TODO
    raise NotImplementedError

def inverses_linear_todo(N):
    # TODO
    raise NotImplementedError

def count_circular_todo(n, m, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một khe, phải "bằng". (2) `C(1,1) = 1`. (3) `3`.

**Bài 2.** (1) `C(3, 2) = 3`. (2) `m = 3`, một khe khác: `2`. (3) `18`.

**Bài 3.** (1) `k = n − 1 − d`. (2) Gọi lại công thức. (3) Kiểm tra `d` ngoài `[0, n−1]` ⇒ 0.

**Bài 4.** (1) `inv[1] = 1`. (2) `inv[i] = (MOD − MOD // i) · inv[MOD % i] mod MOD`. (3) Giai thừa nghịch đảo là tích tiền tố của `inv`.

**Bài 5.** (1) Đếm chuỗi khe vòng `n` khe có đúng `k` dấu `=` và gán giá trị hợp lệ. (2) Với `j = n − k` dấu `≠` trên vòng: số cách gán màu = số cách tô chu trình `j` đỉnh với `m` màu sao cho kề khác nhau: `(m−1)^j + (−1)^j (m−1)` (với `j ≥ 1`; `j = 0` ⇒ `m`). (3) Nhân `C(n, k)`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2, 3
assert Solution().countGoodArrays(2, 3, 1) == 3 == count_brute(2, 3, 1)
assert Solution().countGoodArrays(4, 3, 2) == 18 == count_brute(4, 3, 2)
def count_exact_different(n, m, d):
    return Solution().countGoodArrays(n, m, n - 1 - d) if 0 <= d <= n - 1 else 0
assert all(count_exact_different(n, m, d) == sum(1 for arr in itertools.product(range(m), repeat=n)
           if sum(arr[i] != arr[i - 1] for i in range(1, n)) == d) for n in range(1, 5) for m in range(1, 4) for d in range(n))
print('Bài 1: 3 | Bài 2: 18 | Bài 3 khớp vét cạn.')

Bài 1: 3 | Bài 2: 18 | Bài 3 khớp vét cạn.


In [24]:
# Bài 4, 5
def inverses_linear(N):
    inv = [0, 1] + [0] * (N - 1)
    for i in range(2, N + 1):
        inv[i] = (MOD - MOD // i) * inv[MOD % i] % MOD
    return inv

inv = inverses_linear(1000)
assert all(inv[i] * i % MOD == 1 for i in range(1, 1001))

def count_circular(n, m, k):
    j = n - k                                  # số khe "khác" trên vòng
    if j == 0:
        colorings = m
    elif j == 1:
        colorings = 0                          # một khe khác duy nhất trên vòng: không thể
    else:
        colorings = (pow(m - 1, j, MOD) + (-1) ** j * (m - 1)) % MOD
    return comb(n, k) % MOD * colorings % MOD

def circular_brute(n, m, k):
    return sum(1 for arr in itertools.product(range(m), repeat=n)
               if sum(arr[i] == arr[i - 1] for i in range(n)) == k)

for n in range(2, 6):
    for m in range(1, 4):
        for k in range(n + 1):
            assert count_circular(n, m, k) == circular_brute(n, m, k), (n, m, k)
print('Bài 4: nghịch đảo tuyến tính đúng | Bài 5 khớp vét cạn; n=4, m=3, k=1 ->', count_circular(4, 3, 1))

Bài 4: nghịch đảo tuyến tính đúng | Bài 5 khớp vét cạn; n=4, m=3, k=1 -> 24


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
N = n - 1
fact[0..N];  C = fact[N] * inv(fact[k]) * inv(fact[N-k])
ans = m * pow(m - 1, N - k, MOD) * C  % MOD
```

**Checklist:**

- [ ] `n − 1` khe, không phải `n`?
- [ ] Nhân `m` cho phần tử đầu?
- [ ] Số mũ `n − 1 − k`?
- [ ] Nghịch đảo modulo thay cho `//`?
- [ ] `m = 1` và `k = n − 1` ⇒ 1 (vì `pow(0, 0) = 1`)?